# Churn Prediction — Feature Engineering and Preprocessing

## Objective 

The objective of this notebook is to address the data quality issues and feature engineering decisions identified in `01_eda.ipynb` and prepare the dataset for modeling. 

**Data quality fixes:**
- Replace 11 `NaN` values in `TotalCharges` with `0.0` — the 11 were confirmed as data entry errors (`' '` instead of `0.0`) for brand new customers with `tenure = 0` 
- Drop `customerID` — it carries no predictive information 

**Feature engineering decisions:** 
- Encode binary categorical features (`Yes`/`No` → 0/1) 
- log transform `TotalCharges`
- One-hot encode multi-category features (`Contract`, `InternetService`, `PaymentMethod`) 
- Handle `No internet service` third category in service columns 
- Address redundancy among service features (`OnlineSecurity`, `TechSupport`, `OnlineBackup`, `DeviceProtection`) and between `StreamingTV` and `StreamingMovies` 
- Address confounding between `Contract`, `PaymentMethod`, and `PaperlessBilling` 
- Decide on `tenure` vs `TotalCharges` retention given 0.83 correlation 

Most transformations are fixed mappings applied before the train/test split. Scaling is applied after the split — fitted on training data only and then applied to both train and test to prevent data leakage. 

## Outputs 
- Cleaned, transformed, and scaled dataset saved to `data/processed/` as `X_train.csv`, `X_test.csv`, `y_train.csv`, and `y_test.csv`, ready for modeling in `03_modeling_and_evaluation.ipynb`.

## 2.1 Setup & Load Dataset

Importing some of the same core libraries as `01_eda.ipynb`, with the addition of `os` for directory management, and sklearn modules for train/test splitting and scaling. Path constants and `RANDOM_STATE` are defined here to ensure reproducibility and consistent file references throughout the notebook. `os.makedirs()` is called to ensure the `data/processed/` directory exists before saving — preventing errors when running on a fresh clone of the repo.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore') # suppress sklearn warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
RAW_DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
X_TRAIN_PATH = '../data/processed/X_train.csv'
X_TEST_PATH = '../data/processed/X_test.csv'
Y_TRAIN_PATH = '../data/processed/y_train.csv'
Y_TEST_PATH = '../data/processed/y_test.csv'

os.makedirs('../data/processed/', exist_ok=True)

Loading the raw dataset fresh from `data/raw/` using `RAW_DATA_PATH` to ensure all transformations in this notebook start from the original, unmodified data.

In [17]:
df = pd.read_csv(RAW_DATA_PATH)
print(df.shape)
pd.set_option('display.max_columns', None)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2.2 Target Binarization 

Converting the target variable `Churn` from string values (`Yes`/`No`) to binary (1/0) using `.map()`. This is required since sklearn models expect numeric targets, not string labels.

In [18]:
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})
df['Churn'].head()

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64

`.head()` confirms `Churn` has been successfully converted to binary — `Yes`->`1` and `No`->`0`. The column is now ready to be used as the target variable in modeling.

## 2.3 Dropping `customerID`

`customerID` carries no predictive information — it's a unique identifier for each customer and would only add noise to the model. Dropping it using `.drop()` with `axis=1` to remove the entire column.

In [ ]:
df.drop('customerID', axis=1, inplace=True)
df.head()
df.shape

(7043, 20)

`.head()` and `.shape` confirm `customerID` has been successfully dropped — the column no longer appears in the dataset and `df.shape` returns `(7043, 20)`, confirming 20 remaining columns (19 features + target `Churn`).

## 2.4 Handling Missing Values